# WAXAL ASR — Whisper-small Training (Experiment 001)

Fine-tune `openai/whisper-small` on the WAXAL NLP dataset (Shona) using Google Colab GPU.

**Requirements:**
- Colab GPU runtime (T4 or A100)
- HuggingFace account with access to `google/WaxalNLP`
- Google Drive for checkpoint storage (optional but recommended)

## 1. Environment Setup

In [ ]:
# Check GPU
import torch
gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO GPU"
gpu_mem = torch.cuda.get_device_properties(0).total_mem / 1e9 if torch.cuda.is_available() else 0
print(f"GPU: {gpu_name}")
print(f"VRAM: {gpu_mem:.1f} GB")
assert torch.cuda.is_available(), "Runtime → Change runtime type → T4 GPU"

In [ ]:
# Clone the repo (use dev branch — feature branch has the Whisper implementation)
import os

if not os.path.exists("google-waxal-asr-challenge"):
    !git clone -b feature/whisper-baseline https://github.com/Zeusse-Neumbi/google-waxal-asr-challenge.git
else:
    print("Repo already cloned. Pulling latest...")
    !cd google-waxal-asr-challenge && git pull

%cd google-waxal-asr-challenge

In [ ]:
# Install dependencies (this takes ~3-5 minutes)
# Colab has torch pre-installed, but we need the specific project deps
!pip install -e ".[dev,tracking]" -q
!pip install hf_transfer -q  # faster model downloads

# Verify installation
!python -c "from waxal_asr.metrics import wer; print('waxal_asr imported successfully')"

## 2. HuggingFace Authentication

You need a HuggingFace token with access to the `google/WaxalNLP` dataset.

1. Go to https://huggingface.co/settings/tokens
2. Create a token with `read` permission
3. Add it to Colab Secrets (left sidebar → 🔑 → "HF_TOKEN") or paste below

In [ ]:
# Set HuggingFace token
import os

# Method 1: Colab Secrets (recommended)
try:
    from google.colab import userdata
    hf_token = userdata.get("HF_TOKEN")
    os.environ["HUGGING_FACE_HUB_TOKEN"] = hf_token
    os.environ["HF_TOKEN"] = hf_token
    print("HF token loaded from Colab Secrets.")
except Exception:
    # Method 2: Manual input
    hf_token = input("Enter your HuggingFace token: ")
    os.environ["HUGGING_FACE_HUB_TOKEN"] = hf_token
    os.environ["HF_TOKEN"] = hf_token
    print("HF token set manually.")

# Verify dataset access
from huggingface_hub import HfApi
api = HfApi()
user = api.whoami()
print(f"Logged in as: {user.get('name', 'unknown')}")

## 3. Google Drive (optional — for saving checkpoints)

Mount Drive to persist model checkpoints. Without this, checkpoints are lost when Colab disconnects.

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount("/content/drive")

# Create output directory on Drive
import pathlib
drive_output = pathlib.Path("/content/drive/MyDrive/waxal-asr/outputs")
drive_output.mkdir(parents=True, exist_ok=True)
print(f"Checkpoints will be saved to: {drive_output}")

In [ ]:
# Check available disk space
import shutil
total, used, free = shutil.disk_usage("/")
print(f"Disk: {total / 1e9:.0f} GB total, {used / 1e9:.0f} GB used, {free / 1e9:.0f} GB free")

## 4. Configure Training

**IMPORTANT:** `load_waxal_dataset` always streams from HuggingFace internally (to avoid downloading the 27 GB `unlabeled` split). When `streaming=False`, it materialises the requested subset into a regular `Dataset` in-memory — keeping disk usage bounded by `max_train_samples`, not the full dataset.

In [ ]:
# Keep HF cache on local disk (faster than Drive for model/dataset downloads)
import os
os.environ["HF_HOME"] = "/content/hf_cache"
os.environ["HF_DATASETS_CACHE"] = "/content/hf_cache/datasets"

from waxal_asr.config import load_config

# Load the whisper-small config
cfg = load_config("configs/whisper-small.yaml")

# --- Overrides for Colab ---
# 1. Disable streaming — materialises subset in-memory (always streams from HF internally)
cfg.dataset.streaming = False

# 2. Limit training samples for faster iteration (full dataset: ~15k Shona utterances)
#    Set to None for full training, but start small to verify the pipeline works.
cfg.dataset.max_train_samples = 500  # start small, increase once it works

# 3. Reduce max_steps to match sample count
#    (500 samples / 16 batch_size / 2 grad_accum ≈ 15 steps per epoch)
cfg.training.max_steps = 100  # quick test run — increase to 5000 for real training

# 4. Save to Drive
cfg.paths.outputs = drive_output

# 5. Use fp16 (T4 supports bf16 poorly, fp16 is more reliable)
cfg.model.torch_dtype = "float16"

# Print config summary
print("=" * 60)
print("Training Configuration")
print("=" * 60)
print(f"Model:        {cfg.model.model_id}")
print(f"Language:     {cfg.dataset.language}")
print(f"Streaming:    {cfg.dataset.streaming}")
print(f"Max samples:  {cfg.dataset.max_train_samples}")
print(f"Max steps:    {cfg.training.max_steps}")
print(f"Batch size:   {cfg.training.per_device_train_batch_size}")
print(f"Grad accum:   {cfg.training.gradient_accumulation_steps}")
print(f"Learning rate: {cfg.optimizer.lr}")
print(f"LoRA:         {cfg.lora.enabled}")
print(f"Output dir:   {cfg.paths.outputs}")
print(f"Dtype:        {cfg.model.torch_dtype}")

## 5. Run Training

This will:
1. Download `openai/whisper-small` (~480MB)
2. Load the `google/WaxalNLP` Shona split
3. Fine-tune using `Seq2SeqTrainer`
4. Evaluate WER/CER on the validation set
5. Save the model checkpoint

**Expected time on T4:** ~15-20 min for 100 steps with 500 samples.

In [ ]:
from waxal_asr.utils.seeding import seed_everything
from waxal_asr.training.trainer import Trainer
from waxal_asr.utils.logging import configure_logging, get_logger

# Seed for reproducibility
seed_everything(cfg.repro.seed, deterministic=cfg.repro.deterministic)

# Configure logging
configure_logging(level=cfg.logging.level)

# Create and run trainer
trainer = Trainer(cfg)
trainer.fit()  # dispatches to _fit_whisper() based on model_type="whisper"

## 6. Inference & Submission

After training, generate predictions on the test set and create a submission CSV.

In [ ]:
# Find the saved checkpoint
checkpoint_dir = drive_output / f"whisper-asr-{cfg.dataset.language}" / "checkpoint"
print(f"Checkpoint: {checkpoint_dir}")
print(f"Exists: {checkpoint_dir.exists()}")

# List checkpoint contents
if checkpoint_dir.exists():
    for f in sorted(checkpoint_dir.iterdir()):
        print(f"  {f.name}")

In [ ]:
# Run inference on the test set
from waxal_asr.models.registry import build_model
from waxal_asr.data.dataset import load_waxal_dataset
import torch
import pandas as pd
from tqdm import tqdm

# Build model from config
model_obj = build_model(cfg.model.model_type, config=cfg)
model_obj.load(checkpoint=str(checkpoint_dir))
model = model_obj.model
processor = model_obj.processor
device = model.device
model.eval()

# Load test dataset (non-streaming)
test_ds = load_waxal_dataset(
    dataset_id=cfg.dataset.dataset_id,
    language=cfg.dataset.language,
    split="test",
    streaming=False,
    sample_rate=cfg.dataset.sample_rate,
)

# Convert to list for iteration
print("Loading test samples...")
test_samples = list(test_ds)
print(f"Test samples: {len(test_samples)}")

In [ ]:
# Run inference
import numpy as np

predictions = []
batch_size = 8

for i in tqdm(range(0, len(test_samples), batch_size), desc="Inferring"):
    batch = test_samples[i:i+batch_size]
    
    # Extract audio arrays
    audios = [np.asarray(ex["audio"]["array"]).flatten() for ex in batch]
    
    # Process audio → log-mel features
    inputs = processor(
        audio=audios,
        sampling_rate=16000,
        return_tensors="pt",
        padding=True,
    ).to(device)
    
    # Generate
    with torch.no_grad():
        predicted_ids = model.generate(
            **inputs,
            max_new_tokens=128,
            pad_token_id=processor.tokenizer.pad_token_id,
        )
    
    # Decode
    decoded = processor.batch_decode(predicted_ids, skip_special_tokens=True)
    predictions.extend([t.strip() for t in decoded])

print(f"Generated {len(predictions)} predictions")
print(f"Sample: {predictions[0][:100]}...")

In [ ]:
# Create submission CSV
import pandas as pd

# Get IDs from the dataset
ids = [str(ex.get("id", i)) for i, ex in enumerate(test_samples)]

submission_df = pd.DataFrame({"ID": ids, "Target": predictions})
submission_path = drive_output / "submission_001.csv"
submission_df.to_csv(submission_path, index=False)

print(f"Submission saved: {submission_path}")
print(f"Rows: {len(submission_df)}")
print(submission_df.head())

## 7. Next Steps

- **Increase training:** Set `cfg.dataset.max_train_samples = None` and `cfg.training.max_steps = 5000` for full training
- **Multi-language:** Change `cfg.dataset.language` to `"lin"` (Lingala) or `"lug"` (Luganda)
- **Evaluate locally:** Use `waxal-eval` CLI with references + predictions CSVs
- **Error analysis:** Create `notebooks/05_error_analysis.ipynb` to analyze per-language WER
- **Stronger model:** Try `whisper-medium` or `whisper-large-v3` (needs A100 GPU)